In [1]:
import pandas as pd
import numpy as np
from functions import scrape_nba_data
import matplotlib.pyplot as plt

In [ ]:
raw_df = scrape_nba_data(2021, 2025)
raw_df.head()

Scraping 2024-25...
  569 players
Scraping 2025-26...
  582 players
Scraping 2026-27...
  0 players

Saved 1151 rows to nba_data.csv


/Users/treychase/nets_project/nets-project/functions.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(all_seasons, ignore_index=True)


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,DEF_RATING,NET_RATING,TS_PCT,EFG_PCT,USG_PCT,AST_PCT,REB_PCT,PIE,PACE,SEASON
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,24.0,26,14,12,0.538,...,111.1,-1.4,0.542,0.508,0.189,0.093,0.080,0.082,103.71,2024-25
1,1631260,AJ Green,AJ,1610612749,MIL,25.0,73,44,29,0.603,...,107.6,6.4,0.621,0.612,0.123,0.086,0.051,0.058,100.84,2024-25
2,1642358,AJ Johnson,AJ,1610612764,WAS,20.0,29,8,21,0.276,...,115.6,-11.5,0.480,0.441,0.173,0.178,0.042,0.053,100.83,2024-25
3,203932,Aaron Gordon,Aaron,1610612743,DEN,29.0,51,33,18,0.647,...,113.0,9.5,0.650,0.607,0.186,0.143,0.086,0.104,101.64,2024-25
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,28.0,62,39,23,0.629,...,103.6,6.7,0.594,0.571,0.162,0.142,0.044,0.082,103.09,2024-25


In [3]:
raw_df.describe()

,AGE,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OFF_RATING,DEF_RATING,NET_RATING,TS_PCT,EFG_PCT,USG_PCT,AST_PCT,REB_PCT,PIE,PACE
count,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,...,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,1151.000000,1151.00000,1151.000000,1151.000000
mean,26.270200,0.484857,19.549175,3.292268,7.098697,0.454048,1.047350,2.973414,0.304154,1.397741,...,109.814075,111.844657,-2.030669,0.554997,0.525077,0.178156,0.146442,0.08854,0.089142,101.811712
std,4.217043,0.203406,9.224591,2.305012,4.754537,0.109017,0.886542,2.300270,0.140820,1.394908,...,8.914045,8.859052,12.365885,0.106334,0.112139,0.054769,0.085624,0.04045,0.038659,3.593668
min,19.000000,0.000000,0.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,50.000000,30.800000,-111.400000,0.000000,0.000000,0.000000,0.000000,0.00000,-0.100000,88.670000
25%,23.000000,0.333000,12.100000,1.500000,3.300000,0.411000,0.300000,1.100000,0.272500,0.500000,...,106.600000,108.800000,-7.600000,0.524000,0.492500,0.141000,0.083000,0.05900,0.069000,99.570000
50%,25.000000,0.500000,19.800000,2.800000,6.200000,0.452000,0.900000,2.600000,0.338000,1.000000,...,110.900000,112.200000,-1.500000,0.569000,0.536000,0.170000,0.124000,0.07900,0.087000,101.480000
75%,28.000000,0.623000,27.150000,4.600000,9.650000,0.500000,1.600000,4.400000,0.381000,1.800000,...,114.700000,115.600000,3.850000,0.607000,0.577000,0.210000,0.194500,0.11050,0.110000,103.505000
max,41.000000,1.000000,38.000000,11.800000,22.800000,1.000000,4.400000,11.300000,1.000000,7.900000,...,171.400000,180.000000,85.700000,1.250000,1.250000,0.455000,0.556000,0.25000,0.400000,133.270000


In [4]:
raw_df.dtypes

PLAYER_ID             object
PLAYER_NAME           object
NICKNAME              object
TEAM_ID               object
TEAM_ABBREVIATION     object
                      ...   
AST_PCT              float64
REB_PCT              float64
PIE                  float64
PACE                 float64
SEASON                object
Length: 78, dtype: object

In [5]:
# Some players get traded mid-season: the API gives one row per team plus a
# "TOT" (total) row combining them. Keep only the TOT row when present so
# each player has exactly one row per season. (A groupby(...).apply() here
# would silently drop PLAYER_ID/SEASON under current pandas, since group-by
# columns are excluded from what's passed to the function — so this uses a
# vectorized transform instead.)
has_tot = raw_df.groupby(["PLAYER_ID", "SEASON"])["TEAM_ABBREVIATION"].transform(
    lambda s: (s == "TOT").any()
)
df = raw_df[~has_tot | (raw_df["TEAM_ABBREVIATION"] == "TOT")].reset_index(drop=True)

# eda_box_scores.py expects the scraper's original column names (SEASON,
# PLAYER_ID, FG_PCT, GP, MIN, FGA, ...), so the deduplicated data is saved
# under those names — this is the file the EDA cell below reads.
df.to_csv("nba_data.csv", index=False)

# A friendlier, renamed per-game table for display/export. Not used by the
# EDA script, which relies on the original column names saved above.
id_cols = ["SEASON", "PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "PLAYER_POSITION"]

shooting_cols = ["FG_PCT", "FG3_PCT", "FT_PCT"]

per_game_cols = ["PTS", "REB", "AST", "STL", "TOV"]

advanced_cols = [
    "OFF_RATING", "DEF_RATING", "NET_RATING",
    "TS_PCT", "EFG_PCT", "USG_PCT", "AST_PCT", "REB_PCT", "PIE", "PACE",
]

keep_cols = id_cols + shooting_cols + per_game_cols + advanced_cols
keep_cols = [c for c in keep_cols if c in df.columns]

per_game = df[keep_cols].rename(columns={
    "SEASON": "YEAR",
    "PLAYER_NAME": "PLAYER",
    "TEAM_ABBREVIATION": "TEAM",
    "PTS": "PPG",
    "REB": "RPG",
    "AST": "APG",
    "STL": "SPG",
    "TOV": "TOPG",
})

per_game = per_game.sort_values(["YEAR", "PLAYER"]).reset_index(drop=True)

per_game.to_csv("nba_per_game.csv", index=False)
per_game.head()


,YEAR,PLAYER,TEAM,AGE,FG_PCT,FG3_PCT,FT_PCT,PPG,RPG,APG,...,OFF_RATING,DEF_RATING,NET_RATING,TS_PCT,EFG_PCT,USG_PCT,AST_PCT,REB_PCT,PIE,PACE
0,2024-25,A.J. Lawson,TOR,24.0,0.421,0.327,0.683,9.1,3.3,1.2,...,109.7,111.1,-1.4,0.542,0.508,0.189,0.093,0.080,0.082,103.71
1,2024-25,AJ Green,MIL,25.0,0.429,0.427,0.815,7.4,2.4,1.5,...,114.0,107.6,6.4,0.621,0.612,0.123,0.086,0.051,0.058,100.84
2,2024-25,AJ Johnson,WAS,20.0,0.385,0.267,0.865,7.6,2.0,2.6,...,104.1,115.6,-11.5,0.480,0.441,0.173,0.178,0.042,0.053,100.83
3,2024-25,Aaron Gordon,DEN,29.0,0.531,0.436,0.810,14.7,4.8,3.2,...,122.5,113.0,9.5,0.650,0.607,0.186,0.143,0.086,0.104,101.64
4,2024-25,Aaron Holiday,HOU,28.0,0.437,0.398,0.829,5.5,1.3,1.3,...,110.3,103.6,6.7,0.594,0.571,0.162,0.142,0.044,0.082,103.09


In [6]:
from eda_box_scores import run_eda

run_eda(input_csv="nba_data.csv", output_dir="eda_output")

Loaded 1151 player-season rows, 2 seasons (2024-25 to 2025-26)

=== MISSINGNESS ===
Series([], )

=== DISTRIBUTIONS ===
         FG_PCT   FG3_PCT    FT_PCT    TS_PCT   EFG_PCT       AGE   USG_PCT       MIN        GP
count  1151.000  1151.000  1151.000  1151.000  1151.000  1151.000  1151.000  1151.000  1151.000
mean      0.454     0.304     0.729     0.555     0.525    26.270     0.178    19.549    46.010
std       0.109     0.141     0.195     0.106     0.112     4.217     0.055     9.225    24.642
min       0.000     0.000     0.000     0.000     0.000    19.000     0.000     0.900     1.000
25%       0.411     0.272     0.683     0.524     0.492    23.000     0.141    12.100    24.500
50%       0.452     0.338     0.770     0.569     0.536    25.000     0.170    19.800    51.000
75%       0.500     0.381     0.834     0.607     0.577    28.000     0.210    27.150    68.000
max       1.000     1.000     1.000     1.250     1.250    41.000     0.455    38.000    82.000

18.0% of player

/Users/treychase/nets_project/nets-project/eda_box_scores.py:138: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  binned = valid.groupby(pd.cut(valid["AGE"], bins=range(18, 42, 2)))["TS_PCT"].mean()



=== MULTICOLLINEARITY ===
         FG_PCT  FG3_PCT  FT_PCT  TS_PCT  EFG_PCT
FG_PCT     1.00     0.13    0.12    0.90     0.92
FG3_PCT    0.13     1.00    0.24    0.40     0.41
FT_PCT     0.12     0.24    1.00    0.34     0.21
TS_PCT     0.90     0.40    0.34    1.00     0.97
EFG_PCT    0.92     0.41    0.21    0.97     1.00

Variance Inflation Factors (VIF > ~5 signals a collinearity problem):
 column    VIF
 FG_PCT 218.65
FG3_PCT  12.30
 FT_PCT  23.59
 TS_PCT 676.39
EFG_PCT 638.13

=== AUTOCORRELATION (lag-1 TS%) ===
Lag-1 (consecutive-season) TS% correlation: 0.502
Player-seasons with a valid consecutive prior season: 375 of 944 total (39.7%)

=== VARIANCE DECOMPOSITION ===
Between-player variance share: 84.4%
Within-player (season-to-season) variance share: 15.6%
A large between-player share supports a hierarchical model with player-level intercepts rather than pooling everyone together.

=== SAMPLE SIZE / ATTRITION ===
Mean TS% for player-seasons WITH vs WITHOUT a following season